# EDA for Chat Demo: Metrics for Power BI + Lake EDA

This notebook computes filtered aggregates used by the Power BI visuals (Task 1) and performs exploratory data analysis over the lake data copied from the SQL database (Task 2).

## Imports and Configuration
- Uses `pyodbc` + `pandas`
- Defines output lake locations under `data/metrics/` for aggregates.

In [18]:
# Imports and configuration
import os
from pathlib import Path
import pandas as pd
import numpy as np
import pyodbc

# Lake output base (relative to repo root)
LAKE_BASE = Path("/lakehouse/default/Files/eda")
LAKE_BASE.mkdir(parents=True, exist_ok=True)

print(f"Metrics lake base: {LAKE_BASE}")

StatementMeta(, 0f1ba5ea-07dc-46a9-8dd3-23f68e28c8fe, 20, Finished, Available, Finished)

Metrics lake base: /lakehouse/default/Files/eda


## Load Source Tables (SQL) for Metrics
- Pull minimal columns from `dbo.chat_history` and `dbo.tool_usage` using `pyodbc`.
- Convert `trace_end` to pandas datetime.

In [19]:
# Load source tables (minimal columns) from SQL
cols_chat = [
    "message_id","session_id","trace_id","user_id","message_type",
    "total_tokens","prompt_tokens","completion_tokens",
    "response_time_ms","trace_end","tool_call_id","tool_name"
]
cols_tool = ["tool_call_id","session_id","trace_id","tool_name","status"]

query_chat = f"SELECT {', '.join(cols_chat)} FROM dbo.chat_history"
query_tool = f"SELECT {', '.join(cols_tool)} FROM dbo.tool_usage"

# If your SQL query targets tables in the data lake, Spark is required. 
# If your SQL query targets the data warehouse, pandas can query it directly.
chat_history = spark.sql(query_chat).toPandas()
tool_usage   = spark.sql(query_tool).toPandas()

# Type conversions
chat_history["trace_end"] = pd.to_datetime(chat_history["trace_end"], errors="coerce")
chat_history["message_type"] = chat_history["message_type"].astype("string")
chat_history["tool_name"] = chat_history["tool_name"].astype("string")

tool_usage["tool_name"] = tool_usage["tool_name"].astype("string")
tool_usage["status"] = tool_usage["status"].astype("string")

print("Loaded:")
print(" chat_history:", chat_history.shape)
print(" tool_usage:", tool_usage.shape)
chat_history.head(3)

StatementMeta(, 0f1ba5ea-07dc-46a9-8dd3-23f68e28c8fe, 21, Finished, Available, Finished)

Loaded:
 chat_history: (1000, 12)
 tool_usage: (261, 5)


,message_id,session_id,trace_id,user_id,message_type,total_tokens,prompt_tokens,completion_tokens,response_time_ms,trace_end,tool_call_id,tool_name
0,msg_1c930090d89d409f,session_2546abdb379c,trace_01d2f3b5936e4a5c,user_001,human,NaN,NaN,NaN,NaN,2025-11-05 05:31:26.316861,None,<NA>
1,msg_207b88301da645a0,session_2546abdb379c,trace_01d2f3b5936e4a5c,user_001,ai,535.0,372.0,163.0,872.0,2025-11-05 05:31:27.188861,call_1572ac4f6edf,create_new_account
2,msg_93a58286a49249de,session_2546abdb379c,trace_01d2f3b5936e4a5c,user_001,tool_result,192.0,NaN,NaN,NaN,2025-11-05 05:31:27.355861,call_1572ac4f6edf,create_new_account


## Build Daily Date Columns
Create a `date` column normalized to midnight, using `trace_end` and dropping null dates.

In [20]:
# Build date column
chat_history = chat_history.copy()
chat_history["date"] = chat_history["trace_end"].dt.normalize()

# Drop rows without a date (to align with report slicers)
chat_history = chat_history.dropna(subset=["date"])  # ensures no NaT dates
chat_history["date"] = chat_history["date"].dt.date  # cast to date for PBI friendliness

print("Min/Max date:", chat_history["date"].min(), chat_history["date"].max())
chat_history.head(3)

StatementMeta(, 0f1ba5ea-07dc-46a9-8dd3-23f68e28c8fe, 22, Finished, Available, Finished)

Min/Max date: 2025-09-22 2025-12-15


,message_id,session_id,trace_id,user_id,message_type,total_tokens,prompt_tokens,completion_tokens,response_time_ms,trace_end,tool_call_id,tool_name,date
0,msg_1c930090d89d409f,session_2546abdb379c,trace_01d2f3b5936e4a5c,user_001,human,NaN,NaN,NaN,NaN,2025-11-05 05:31:26.316861,None,<NA>,2025-11-05
1,msg_207b88301da645a0,session_2546abdb379c,trace_01d2f3b5936e4a5c,user_001,ai,535.0,372.0,163.0,872.0,2025-11-05 05:31:27.188861,call_1572ac4f6edf,create_new_account,2025-11-05
2,msg_93a58286a49249de,session_2546abdb379c,trace_01d2f3b5936e4a5c,user_001,tool_result,192.0,NaN,NaN,NaN,2025-11-05 05:31:27.355861,call_1572ac4f6edf,create_new_account,2025-11-05


## “% of User Questions answered by tool usage”
Filter: `message_type` is not `human` and not `tool_result`. Numerator = rows with tool (non-null `tool_call_id` or present in `tool_usage` by `trace_id`). Denominator = all filtered rows. Grouped daily.

In [21]:
# Compute % of user questions answered by tool usage
ai_like = chat_history[~chat_history["message_type"].isin(["human", "tool_result"])]

# Determine which traces used a tool (by tool_call_id on row OR matching trace_id in tool_usage)
has_tool_direct = ai_like["tool_call_id"].notna()
has_tool_via_usage = ai_like["trace_id"].isin(tool_usage["trace_id"].dropna().unique())
ai_like = ai_like.assign(has_tool = (has_tool_direct | has_tool_via_usage))

# Daily aggregates
daily_ai = ai_like.groupby("date", as_index=False).agg(
    ai_total=("trace_id", "count"),
    ai_with_tool=("has_tool", "sum")
)
daily_ai["pct_ai_answered_by_tool"] = (
    daily_ai["ai_with_tool"].astype(float) / daily_ai["ai_total"].replace(0, np.nan)
)

# Overall summary
overall_ai = pd.DataFrame({
    "ai_total": [int(ai_like.shape[0])],
    "ai_with_tool": [int(ai_like["has_tool"].sum())]
})
overall_ai["pct_ai_answered_by_tool"] = (
    overall_ai["ai_with_tool"].astype(float) / overall_ai["ai_total"].replace(0, np.nan)
)

print("Daily % rows:", daily_ai.shape)
print("Overall %:")
daily_ai.head(3)

StatementMeta(, 0f1ba5ea-07dc-46a9-8dd3-23f68e28c8fe, 23, Finished, Available, Finished)

Daily % rows: (45, 4)
Overall %:


,date,ai_total,ai_with_tool,pct_ai_answered_by_tool
0,2025-09-22,12,10,0.833333
1,2025-09-24,6,1,0.166667
2,2025-09-25,23,17,0.739130


## “Token Usage by Message Type” (exclude `ai`)
Group by `date` and `message_type`, summing `total_tokens`, excluding `ai`. Provide a pivot for quick validation.

In [22]:
# Token usage by message type (exclude 'ai')
df_non_ai = chat_history[chat_history["message_type"] != "ai"].copy()
df_non_ai["total_tokens"] = df_non_ai["total_tokens"].fillna(0)

agg_tokens = (
    df_non_ai.groupby(["date", "message_type"], as_index=False)["total_tokens"].sum()
)

pivot_tokens = agg_tokens.pivot(index="date", columns="message_type", values="total_tokens").fillna(0)
print("Aggregated token usage (long):", agg_tokens.shape)
agg_tokens.head(3)

StatementMeta(, 0f1ba5ea-07dc-46a9-8dd3-23f68e28c8fe, 24, Finished, Available, Finished)

Aggregated token usage (long): (89, 3)


,date,message_type,total_tokens
0,2025-09-22,human,0.0
1,2025-09-22,tool_result,1162.0
2,2025-09-24,human,0.0


## Tool Health for Selected Tools
Filter to `{'create_new_account','get_transactions_summary','get_user_accounts','transfer_money'}` and compute counts by `status` and error rates.

In [23]:
# Tool Health for selected tools
selected_tools = {"create_new_account","get_transactions_summary","get_user_accounts","transfer_money"}

th = tool_usage[tool_usage["tool_name"].isin(selected_tools)].copy()
th["status"] = th["status"].fillna("unknown").astype("string")

# Counts by tool and status
health_counts = (
    th.groupby(["tool_name","status"], as_index=False)
      .agg(calls=("trace_id","count"))
)

# Error rates per tool: errored / total per tool
per_tool = th.groupby("tool_name", as_index=False).agg(total=("trace_id","count"))
errors = (
    th[th["status"].str.lower().isin(["error","failed","failure"])]
      .groupby("tool_name", as_index=False)
      .agg(errored=("trace_id","count"))
)
per_tool = per_tool.merge(errors, on="tool_name", how="left").fillna({"errored":0})
per_tool["error_rate_per_tool"] = per_tool["errored"].astype(float) / per_tool["total"].replace(0, np.nan)

print("Health counts:", health_counts.shape)
health_counts.head(3)

StatementMeta(, 0f1ba5ea-07dc-46a9-8dd3-23f68e28c8fe, 25, Finished, Available, Finished)

Health counts: (4, 3)


,tool_name,status,calls
0,create_new_account,completed,56
1,get_transactions_summary,completed,50
2,get_user_accounts,completed,61


## Persist Aggregates to Lake (CSV/Parquet) and Quick Validation
Write outputs to `data/metrics/` with date-friendly types for Power BI. Then print heads and row counts.

In [24]:
# Persist aggregates to lake
from datetime import datetime
run_ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

# Create versioned folders
pct_dir = LAKE_BASE / "pct_ai_with_tool" / run_ts
tokens_dir = LAKE_BASE / "token_usage_by_message_type" / run_ts
health_dir = LAKE_BASE / "tool_health" / run_ts
for d in [pct_dir, tokens_dir, health_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Write
(daily_ai.assign(date=pd.to_datetime(daily_ai["date"]))
 .to_parquet(pct_dir / "daily_pct.parquet", index=False))
(daily_ai.assign(date=pd.to_datetime(daily_ai["date"]))
 .to_csv(pct_dir / "daily_pct.csv", index=False))

overall_ai.to_parquet(pct_dir / "overall_pct.parquet", index=False)
overall_ai.to_csv(pct_dir / "overall_pct.csv", index=False)

(agg_tokens.assign(date=pd.to_datetime(agg_tokens["date"]))
 .to_parquet(tokens_dir / "tokens.parquet", index=False))
(agg_tokens.assign(date=pd.to_datetime(agg_tokens["date"]))
 .to_csv(tokens_dir / "tokens.csv", index=False))

health_counts.to_parquet(health_dir / "health_counts.parquet", index=False)
health_counts.to_csv(health_dir / "health_counts.csv", index=False)
per_tool.to_parquet(health_dir / "per_tool.parquet", index=False)
per_tool.to_csv(health_dir / "per_tool.csv", index=False)

print("Wrote:")
print(" ", pct_dir)
print(" ", tokens_dir)
print(" ", health_dir)

# Quick validation
assert daily_ai.shape[0] > 0, "No daily rows for % answered by tool"
assert agg_tokens.shape[0] > 0, "No rows for token usage (non-ai)"
assert set(health_counts["tool_name"]) <= {"create_new_account","get_transactions_summary","get_user_accounts","transfer_money"}, "Unexpected tool names present"

print("Validation complete.")

StatementMeta(, 0f1ba5ea-07dc-46a9-8dd3-23f68e28c8fe, 26, Finished, Available, Finished)

Wrote:
  /lakehouse/default/Files/eda/pct_ai_with_tool/20251218T060646Z
  /lakehouse/default/Files/eda/token_usage_by_message_type/20251218T060646Z
  /lakehouse/default/Files/eda/tool_health/20251218T060646Z
Validation complete.
